# 05 - Curadoria Taxonômica ICTV, Saneamento Híbrido e Auditoria de Proveniência

Este notebook reproduz a integração entre a taxonomia do Kraken2 (`species_name_ids.csv`),
a taxonomia do NCBI e a classificação oficial do ICTV (`ICTV_Master_Species_List_2025_MSL41.v1.xlsx`),
saneando a taxonomia híbrida (`main_taxonomy_hybrid_CLEANSED.parquet`) e realizando as auditorias
de proveniência e viabilidade de split por genoma (`viral_cnn.data.audit` e `viral_cnn.data.grouped_split`).

**Invariantes científicas (`AD-001`, `AD-002`, `AD-007`):**
- 13.683 espécies virais fechadas com mapeamento 0-based `model_label = id - 1`.
- Encadeamento dos 8 níveis hierárquicos: Realm, Kingdom, Phylum, Class, Order, Family, Genus, Species.
- Split desincorporado de genoma requer $\ge 2$ genomas por espécie (`AD-007`).


In [7]:
import os
import sys
import json
from pathlib import Path

# Resolução autônoma de caminho para JupyterHub remoto e desenvolvimento local
current_dir = Path.cwd().resolve()
for r in [current_dir, current_dir.parent, current_dir.parent.parent, current_dir.parent.parent.parent]:
    if (r / "src" / "viral_cnn").exists():
        if str(r / "src") not in sys.path:
            sys.path.insert(0, str(r / "src"))
        break

import pandas as pd
import numpy as np
import pyarrow.parquet as pq

from viral_cnn.notebook_runtime import (
    setup_notebook_runtime,
    ensure_smoke_test_artifacts,
    write_notebook_manifest,
    resolve_split_files,
)
from viral_cnn.taxonomy.curation import (
    build_hybrid_taxonomy,
    build_ictv_rename_map,
    generate_synthetic_taxonomy,
    match_ictv_taxonomy,
    resolve_local_ictv_taxonomy,
    sanitize_hybrid_taxonomy,
)
from viral_cnn.taxonomy.mapping import load_taxonomy_mapping
from viral_cnn.data.audit import audit_partition_duplicates
from viral_cnn.data.grouped_split import build_genome_disjoint_split

# 1. Inicialização do runtime e contexto de execução (AD-017)
ctx = setup_notebook_runtime()
output_dir = Path("results_reproduction/taxonomy")
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Device: {ctx.device}")
print(f"Smoke Test Mode: {ctx.is_smoke_test}")
print(f"Data Root: {ctx.data_root}")
print(f"Output Dir: {output_dir}")


Device: cuda
Smoke Test Mode: False
Data Root: /home/mlima/workspace
Output Dir: results_reproduction/taxonomy


In [19]:
# 2. Curadoria da Taxonomia Híbrida ICTV / NCBI
ictv_master_path = Path("/home/mlima/workspace/cnn_kraken/ICTV_Master_Species_List_2025_MSL41.v1.xlsx")
species_csv_path = Path(ctx.topology.paths["short"]["species_csv"])

if ctx.is_smoke_test or not ictv_master_path.exists():
    print("[SMOKE_TEST] Generating deterministic synthetic taxonomy dataset...")
    taxonomy_df = generate_synthetic_taxonomy(num_classes=100)
else:
    print(f"[ICTV] Loading ICTV Master Species List from: {ictv_master_path}")
    ictv_master = pd.read_excel(ictv_master_path, sheet_name="MSL")
    ictv_renames = pd.read_excel(ictv_master_path, sheet_name="Taxa Renamed or Abolished")
    species_labels = pd.read_csv(species_csv_path)
    taxonomy_df = build_hybrid_taxonomy(
        species_labels=species_labels,
        ictv_master=ictv_master,
        ictv_renames=ictv_renames,
    )

# Saneamento e validação de invariantes (AD-002: model_label = id - 1)
taxonomy_df = sanitize_hybrid_taxonomy(taxonomy_df)
out_parquet = output_dir / "main_taxonomy_hybrid_CLEANSED.parquet"
taxonomy_df.to_parquet(out_parquet, index=False)
print(f"[SAVED] Cleaned taxonomy saved to {out_parquet} (Shape: {taxonomy_df.shape})")
print("\nDistribuição por nível taxonômico:")
for rank in ["Realm", "Kingdom", "Phylum", "Class", "Order", "Family", "Genus"]:
    if rank in taxonomy_df.columns:
        print(f"  - {rank}: {taxonomy_df[rank].nunique()} classes únicas")


[ICTV] Loading ICTV Master Species List from: /home/mlima/workspace/cnn_kraken/ICTV_Master_Species_List_2025_MSL41.v1.xlsx
[SAVED] Cleaned taxonomy saved to results_reproduction/taxonomy/main_taxonomy_hybrid_CLEANSED.parquet (Shape: (13683, 17))

Distribuição por nível taxonômico:
  - Realm: 11 classes únicas
  - Kingdom: 12 classes únicas
  - Phylum: 20 classes únicas
  - Class: 42 classes únicas
  - Order: 69 classes únicas
  - Family: 221 classes únicas
  - Genus: 2321 classes únicas


In [20]:
# 3. Auditoria de Duplicatas e Integridade de Partições (viral_cnn.data.audit)
species_parquet_dir = Path(ctx.topology.paths["short"]["species_parquet_dir"])
if ctx.is_smoke_test or not species_parquet_dir.exists():
    print("[SMOKE_TEST] Synthesizing test partition for audit...")
    ensure_smoke_test_artifacts(ctx.data_root, num_samples_per_split=16)

test_files = resolve_split_files(species_parquet_dir, "test")
pf = pq.ParquetFile(test_files[0])
seq_col = "sequence_padded" if "sequence_padded" in pf.schema_arrow.names else "sequence"

audit_results = audit_partition_duplicates(
    file_path=test_files,
    seq_col=seq_col,
    label_col="label",
    sample_size=100 if ctx.is_smoke_test else 50000,
)
print(f"Total de sequências analisadas: {audit_results['total_sequences']}")
print(f"Sequências canônicas únicas: {audit_results['unique_canonical_sequences']}")
print(f"Taxa de duplicatas intra-partição: {audit_results['duplication_rate_pct']:.4f}%")


Total de sequências analisadas: 50000
Sequências canônicas únicas: 49999
Taxa de duplicatas intra-partição: 0.0020%


In [21]:
# 4. Avaliação de Viabilidade do Split Desincorporado de Genoma (AD-007)
# Mock / synthetic representation of species-genome pairing to evaluate split constraints
mock_records = []
num_species = min(len(taxonomy_df), 50)
for sp_idx in range(num_species):
    # Metade das espécies possui genoma único e metade possui 3 genomas
    num_genomes = 3 if sp_idx % 2 == 0 else 1
    for g_idx in range(num_genomes):
        mock_records.append({
            "species_id": sp_idx + 1,
            "genome_id": f"NC_{sp_idx}_{g_idx:03d}",
            "sequence_id": f"frag_{sp_idx}_{g_idx}",
        })

split_result = build_genome_disjoint_split(
    records=mock_records,
    group_col="genome_id",
    class_col="species_id",
    test_size=0.2,
    min_genomes_per_class=2,
    single_genome_policy="train_only",
)
train_parts = split_result["partitions"]["train"]
test_parts = split_result["partitions"]["test"]
print("Auditoria de Viabilidade de Split Agrupado (Genome-Disjoint):")
print(f"  - Total de registros: {len(mock_records)}")
print(f"  - Treino: {len(train_parts)} registros")
print(f"  - Teste: {len(test_parts)} registros")
print(f"  - Espécies avaliáveis no teste (>= 2 genomas): {len(set(r['species_id'] for r in test_parts))}")


Auditoria de Viabilidade de Split Agrupado (Genome-Disjoint):
  - Total de registros: 100
  - Treino: 75 registros
  - Teste: 25 registros
  - Espécies avaliáveis no teste (>= 2 genomas): 25


In [22]:
# 5. Emissão do Manifesto de Execução Rastreável
manifest = write_notebook_manifest(
    notebook_id="05_ictv_taxonomy_curation",
    data_root=ctx.data_root,
    is_smoke_test=ctx.is_smoke_test,
    output_json_path=output_dir / "manifest.json",
    input_files=[str(out_parquet)],
    metrics_summary={
        "num_species": len(taxonomy_df),
        "duplication_rate_pct": audit_results["duplication_rate_pct"],
        "unique_canonical_sequences": audit_results["unique_canonical_sequences"],
        "genome_disjoint_test_classes": len(set(r["species_id"] for r in split_result["partitions"]["test"])),
    },
)
print("\nManifesto de Execução salvo com sucesso:")
print(json.dumps(manifest.metrics_summary, indent=2))



Manifesto de Execução salvo com sucesso:
{
  "num_species": 13683,
  "duplication_rate_pct": 0.002,
  "unique_canonical_sequences": 49999,
  "genome_disjoint_test_classes": 25
}


PosixPath('cnn_kraken/ICTV_Master_Species_List_2025_MSL41.v1.xlsx')

/home/mlima/workspace/cnn_metagenome/notebooks/reproduction/short
